# 06 — DINOv2 Representations for Anomaly Detection

**Pipeline Sentinel — Computer Vision + ETL**

DINOv2 is most useful here as a representation engine: turn visual crops into vectors and ask whether an observation resembles normal activity.

> **Defensive training scope:** sensing, detection, tracking, anomaly scoring, and human-facing alerts for a fictional pipeline corridor. No automated engagement or weapons logic.

## Learning objectives

- Separate semantic representation from object detection.
- Build a normal-activity embedding reference set.
- Score observations by distance from normal embeddings.
- Use an offline HOG representation as a control experiment.

In [ ]:
from pathlib import Path
import sys
HERE=Path.cwd().resolve()
ROOT=HERE.parent if HERE.name=='notebooks' else HERE
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print('Project root:',ROOT)

In [ ]:
import cv2,pandas as pd,numpy as np,matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_distances
from src.embeddings import HOGEmbedder,DinoV2Embedder
from src.utils import crop_xyxy
video=ROOT/'data/raw/pipeline_demo_eo.mp4'; gt=pd.read_csv(ROOT/'data/raw/pipeline_demo_eo_gt.csv')

## 1. Build reference crops

For this lesson, `normal_maintenance` is our reference activity. Real deployments need normal samples across time of day, seasons, authorized vehicle types, crew appearance, sensor angle, and weather.

In [ ]:
cap=cv2.VideoCapture(str(video)); frames=[]
while True:
    ok,frame=cap.read()
    if not ok: break
    frames.append(frame)
cap.release()
normal_rows=gt[gt.scenario_role=='normal_maintenance'].iloc[::5]
probe_rows=gt[gt.scenario_role.isin(['intrusion_vehicle','dismount_loiter','small_uas_like'])].iloc[::8]
def rows_to_crops(rows):
    out=[]
    for _,r in rows.iterrows():
        crop=crop_xyxy(frames[int(r.frame_number)],(r.x1,r.y1,r.x2,r.y2),pad=6)
        if crop.size: out.append(crop)
    return out
normal_crops=rows_to_crops(normal_rows); probe_crops=rows_to_crops(probe_rows)
print(len(normal_crops),'normal reference crops;',len(probe_crops),'probe crops')

## 2. First run a cheap representation

This is the control experiment. If anomaly scoring works with HOG, DINOv2 has to beat it—not merely be newer.

Implementation note: this course uses `skimage.feature.hog` for the HOG baseline rather than `cv2.HOGDescriptor`, because OpenCV Python builds differ in whether that class is exposed.


In [ ]:
embedder=HOGEmbedder(size=(128,128))
normal_z=embedder.encode(normal_crops); probe_z=embedder.encode(probe_crops)
centroid=normal_z.mean(axis=0,keepdims=True)
normal_scores=cosine_distances(normal_z,centroid).ravel(); probe_scores=cosine_distances(probe_z,centroid).ravel()
threshold=np.quantile(normal_scores,.95)
print('95th percentile threshold:',round(float(threshold),4))
print('normal median:',round(float(np.median(normal_scores)),4)); print('probe median:',round(float(np.median(probe_scores)),4))

In [ ]:
plt.figure(figsize=(8,4)); plt.hist(normal_scores,bins=12,alpha=.6,label='normal reference'); plt.hist(probe_scores,bins=12,alpha=.6,label='probe'); plt.axvline(threshold,linestyle='--',label='95% normal threshold'); plt.xlabel('Cosine distance from normal centroid'); plt.ylabel('count'); plt.legend();

## 3. Swap the representation backend to DINOv2

The same experiment now uses a stronger pretrained representation. xFormers is not required for the conceptual workflow; this wrapper uses CPU if CUDA is unavailable. First use may require internet or cached model files.

In [ ]:
RUN_DINOV2=False
if RUN_DINOV2:
    try:
        dino=DinoV2Embedder(model_name='dinov2_vits14')
        normal_dino=dino.encode(normal_crops); probe_dino=dino.encode(probe_crops)
        c=normal_dino.mean(axis=0,keepdims=True)
        dino_normal_scores=cosine_distances(normal_dino,c).ravel(); dino_probe_scores=cosine_distances(probe_dino,c).ravel()
        print('DINO device:',dino.device); print('embedding dim:',normal_dino.shape[1]); print('normal median:',np.median(dino_normal_scores)); print('probe median:',np.median(dino_probe_scores))
    except Exception as e: print('DINOv2 unavailable:',e)
else:
    print('Set RUN_DINOV2=True when ready.')

## What DINO is *not* doing

DINOv2 is not being handed our CSV labels and memorizing `truck`, `person`, or `drone`. It learned broadly useful visual structure during self-supervised pretraining. We use our curated examples to **evaluate or organize those embeddings**.